## Bag of Words
Amaç:
- Metin temsili: metin listesi -> sayısal vektörlere çevir
- sklearn CountVectorizer: kelimelerin kaç defa geçtiğini sayar ve vektör temsiline dönüştürürür

Sonuç:
- kelime kümesi (vocabulary)
- her metin listesi sayısal vektörler ile temsil edilecek

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# örnek metinlerden oluşan küçük bir veri seti oluştur.
dokumanlar = [
    "kedi bahçede",
    "kedi evde"
]

In [ ]:
# count vektorizer nesnesini oluştur
kelime_sayac = CountVectorizer()

In [ ]:
#dokumanları sayısal vektörlere çevir (bag of words uygula)
dokuman_vektorleri = kelime_sayac.fit_transform(dokumanlar)
dokuman_vektorleri

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4 stored elements and shape (2, 3)>

In [ ]:
# count vektorizer tarafından bulunan kelime listesi yan vocabulary oluşturalım print ettirelim
kelime_kumesi = kelime_sayac.get_feature_names_out()
print(f"Kelime Kümesi: {kelime_kumesi}")

Kelime Kümesi ['bahçede' 'evde' 'kedi']


In [ ]:
# vektor temsiline bakalım
vektor_temsili = dokuman_vektorleri.toarray()
print(f"Vektör temsili: \n{vektor_temsili}")

Vektör temsili: 
[[1 0 1]
 [0 1 1]]


## Bag of Words
Amaç:
- IMDB film yorumları içeren veri seti ile bag of words
- csv dosyasından veriyi oku (IMDB Dataset.csv)
- text cleaning (küçük büyük harf çevirme, rakam ve özel karakterlerden kurtulma)
- metinleri sayısal vektörlere dönüştür
- kelime frekanslarını hesapla ve en sık geçen 5 kelimeyi listele

In [ ]:
# gerekli kütüphanelerin içeri aktarılması
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer # bag of words
import re # regular expression (veri temizleme)
from collections import Counter

In [ ]:
# read.csv
veri = pd.read_csv("/content/sample_data/IMDB Dataset.csv")
veri.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
# yorumları (reivew) al
yorumlar = veri["review"]
yorumlar

,review
0,One of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...
2,I thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is..."
...,...
49995,I thought this movie did a down right good job...
49996,"Bad plot, bad dialogue, bad acting, idiotic di..."
49997,I am a Catholic taught in parochial elementary...
49998,I'm going to have to disagree with the previou...


In [ ]:
#etiketleri (sentiment) al
etiketler = veri["sentiment"]
etiketler

,sentiment
0,positive
1,positive
2,positive
3,negative
4,positive
...,...
49995,positive
49996,negative
49997,negative
49998,negative


In [ ]:
# metin temizleme
def metin_temizle(metin):
  #tüm harfleri küçük harfe çevir
  metin = metin.lower()

  #rakamları kaldır
  metin = re.sub(r"\d+", "",metin)

  #özel karakterleri kaldır
  metin = re.sub(r"[^\w\s]","",metin)

  #çok kısa kelimeleri yani 2 karakterden kısa olanları kaldır
  metin = " ".join([kelime for kelime in metin.split() if len(kelime) > 2])

  #temizlenmiş veriyi return et
  return metin

In [ ]:
#yorumları temizleme işlemi uygula
temizlenmiş_yorumlar = [metin_temizle(y) for y in yorumlar]

In [ ]:
# bow
bow_modeli = CountVectorizer()

In [ ]:
# ilk 75 yorumu sayısal vektörlere çevirelim
yorum_vektorleri = bow_modeli.fit_transform(temizlenmiş_yorumlar[:75])

In [ ]:
# vocabulary (kelime kümesi)
kelime_kumesi = bow_modeli.get_feature_names_out()
kelime_kumesi


array(['abbot', 'abetted', 'abiding', ..., 'zone', 'zooms', 'zwick'],
      dtype=object)

In [ ]:
vektor_temsili = yorum_vektorleri.toarray()
vektor_temsili

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
# her kelimenin toplam kaç adet geçtiği
kelime_sayilari = yorum_vektorleri.sum(axis=0).A1

In [ ]:
#kelimeler ile frekansları bir sözlükte eşleştirelim
kelime_frekansi = dict(zip(kelime_kumesi, kelime_sayilari))
en_cok_gecen_5_kelime = Counter(kelime_frekansi).most_common(5)
en_cok_gecen_5_kelime

[('the', np.int64(1033)),
 ('and', np.int64(463)),
 ('this', np.int64(185)),
 ('that', np.int64(174)),
 ('with', np.int64(132))]

## TF-IDF
Örnek birkaç cümle üzerinden TF-IDF uygulayarak cümleleri vektörleştirmek

Adımlar:
1. Küçük bir belge oluştur
2. TF-IDF vektörizer ile belgeleri sayısal vektörlere dönüştür
3. Kelime kümesini oluştur
4. Belgelerin tf-idf vektör temsillerini elde edelim
5. Tüm belgeler için kelimelerin ortalama tf-idf değerlerini hesaplayalım

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# örnek belgeleri tanımla
belgeler = [
    "Köpek çok tatlı bir hayvandır",
    "Köpek ve kuşlar çok tatlı hayvanlardır",
    "İnekler süt üretirler"
]

In [ ]:
# tf idf modeli oluştur
tfidf_modeli = TfidfVectorizer()

In [ ]:
# belgeleri sayısal vektörlere dönüştür
belge_vektorleri = tfidf_modeli.fit_transform(belgeler)

In [ ]:
# kelime kümesini yani vocab oluştur
kelime_kumesi = tfidf_modeli.get_feature_names_out()

In [ ]:
# belgelerin tf idf değerlerini numpy formatına çevir
vektor_temsili = belge_vektorleri.toarray()

In [ ]:
vektor_temsili

array([[0.51741994, 0.51741994, 0.        , 0.        , 0.3935112 ,
        0.        , 0.        , 0.3935112 , 0.        , 0.3935112 ,
        0.        ],
       [0.        , 0.        , 0.45954803, 0.45954803, 0.34949812,
        0.        , 0.        , 0.34949812, 0.45954803, 0.34949812,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.57735027, 0.57735027, 0.        , 0.        , 0.        ,
        0.57735027]])

In [ ]:
# bunları okunabilir hale getir
df_tf_idf = pd.DataFrame(vektor_temsili, columns=kelime_kumesi)
df_tf_idf

,bir,hayvandır,hayvanlardır,kuşlar,köpek,nekler,süt,tatlı,ve,çok,üretirler
0,0.51742,0.51742,0.000000,0.000000,0.393511,0.00000,0.00000,0.393511,0.000000,0.393511,0.00000
1,0.00000,0.00000,0.459548,0.459548,0.349498,0.00000,0.00000,0.349498,0.459548,0.349498,0.00000
2,0.00000,0.00000,0.000000,0.000000,0.000000,0.57735,0.57735,0.000000,0.000000,0.000000,0.57735


In [ ]:
# her kelimenin belgeler arasındaki ortalama tf idf değerlerini hesapla
ortalama_tf_idf = df_tf_idf.mean(axis=0)
ortalama_tf_idf

,0
bir,0.172473
hayvandır,0.172473
hayvanlardır,0.153183
kuşlar,0.153183
köpek,0.247670
nekler,0.192450
süt,0.192450
tatlı,0.247670
ve,0.153183
çok,0.247670
